# RAG + LLM Script Generator

In this notebook you will build a Retrieval Augmented Generation system that answers one question:

> **Given an invoice and its predicted payment status — what should we do?**

The system retrieves the relevant sections from our company playbook and uses Gemini to generate a structured action script: which email template to send, the tone, the priority, and the full email draft.

## Setup

Run the cell below to import libraries and load the API key.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import json
import re
from pathlib import Path
from datetime import datetime
from pprint import pprint
from IPython.display import Markdown, display
import pandas as pd


from dotenv import load_dotenv
load_dotenv("../.env")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Add GEMINI_API_KEY to your .env file!"

PLAYBOOK_PATH = Path("../data/playbook/")
CHROMA_PATH   = Path("../data/chroma_db/")

print("Setup complete.")

Setup complete.


## Why RAG?

An LLM on its own does not know our company's payment policies.

That is where RAG steps in:
1. **Retrieval** — find the most relevant sections of our playbook for a given invoice situation
2. **Augmented** — add those sections to the prompt as context
3. **Generation** — Gemini generates the correct action based on the playbook, not just its training data

This means the system is always grounded in our actual company policy — and we can update the playbook without retraining any model.

## Our data — the playbook

Our knowledge base consists of 4 markdown documents in `data/playbook/`:

| File | Content |
|------|---------|
| `01_payment_communication_policy.md` | Trigger timeline, general principles, customer risk tiers |
| `02_email_templates.md` | 7 email templates, one per stage |
| `03_decision_rules.md` | Rules for selecting the right action per invoice |
| `04_faq_edge_cases.md` | Edge cases: disputes, partial payments, new customers |

Run the cell below to see what files we have:


In [2]:
for f in sorted(PLAYBOOK_PATH.glob("*.md")):
    size = f.stat().st_size
    lines = len(f.read_text().split("\n"))
    print(f"  {f.name}  ({lines} lines, {size:,} bytes)")

  01_payment_communication_policy.md  (63 lines, 2,646 bytes)
  02_email_templates.md  (194 lines, 5,146 bytes)
  03_decision_rules.md  (94 lines, 3,083 bytes)
  04_faq_edge_cases.md  (64 lines, 3,111 bytes)


## Embedding documents

Embedding a document means converting text into a vector of numbers that represents its meaning.

Documents with similar meaning end up close to each other in vector space — that is what makes retrieval work.

Let's instantiate the embedder first and try it out:

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY,
)

Try the embedder on a sample query:

In [4]:
sample_embedding = embeddings.embed_query("Invoice overdue by 5 days. What action should we take?")

print("Type:            ", type(sample_embedding))
print("Embedding size:  ", len(sample_embedding))
print("First 5 values:  ", sample_embedding[:5])

Type:             <class 'list'>
Embedding size:   3072
First 5 values:   [0.0048789564, 0.028552927, -0.0032335774, -0.06177467, 0.017421192]


Each query becomes a list of numbers — the embedding size tells us how many dimensions the vector space has.

When we search for similar documents, we compare this vector to the vectors of all our playbook chunks and return the closest ones.

## Load and split the playbook

Our playbook files are markdown documents structured with headers (`#`, `##`, `###`).

We use `MarkdownHeaderTextSplitter` to split by section — this keeps each chunk semantically coherent. A chunk about "Stage 4: First Overdue Notice" stays together as a unit.



In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split = [
    ("#",   "section_h1"),
    ("##",  "section_h2"),
    ("###", "section_h3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split,
    strip_headers=False,
)

all_docs = []
for md_file in sorted(PLAYBOOK_PATH.glob("*.md")):
    text = md_file.read_text(encoding="utf-8")
    docs = splitter.split_text(text)
    for doc in docs:
        doc.metadata["source"] = md_file.name
    all_docs.extend(docs)
    print(f"  {md_file.name}: {len(docs)} chunks")

print(f"\nTotal chunks: {len(all_docs)}")

  01_payment_communication_policy.md: 7 chunks
  02_email_templates.md: 8 chunks
  03_decision_rules.md: 12 chunks
  04_faq_edge_cases.md: 7 chunks

Total chunks: 34


Let's explore the chunks:

In [6]:
print("Type of all_docs:     ", type(all_docs))
print("Number of chunks:     ", len(all_docs))
print("Type of one chunk:    ", type(all_docs[0]))
print("Total characters:     ", sum(len(d.page_content) for d in all_docs))

Type of all_docs:      <class 'list'>
Number of chunks:      34
Type of one chunk:     <class 'langchain_core.documents.base.Document'>
Total characters:      13957


In [7]:
# Inspect a specific chunk 
chunk = all_docs[11]  # Stage 4: First Overdue Notice

print("Metadata:")
pprint(chunk.metadata)
print("\nContent:")
display(Markdown(chunk.page_content))

Metadata:
{'section_h1': 'Payment Communication — Email Templates',
 'section_h2': 'Stage 4: First Overdue Notice (5 days past due)',
 'source': '02_email_templates.md'}

Content:


## Stage 4: First Overdue Notice (5 days past due)  
**Subject:** Overdue payment notice — Invoice [INVOICE_ID]  
**Body:**  
Dear [CUSTOMER_NAME],  
We notice that invoice [INVOICE_ID] for [AMOUNT], which was due on [DUE_DATE], remains unpaid.  
We understand that oversights can happen and wanted to bring this to your attention. Please arrange payment at your earliest convenience.  
- Invoice number: [INVOICE_ID]
- Amount due: [AMOUNT]
- Original due date: [DUE_DATE]
- Days overdue: 5  
If payment has already been made, please send us confirmation so we can update our records.  
If you are experiencing difficulties, we encourage you to contact us to discuss options.  
Kind regards,
[AR_TEAM_NAME]
Accounts Receivable
[COMPANY_NAME]
[AR_EMAIL] | [AR_PHONE]  
---

# Summary table of all chunks
chunk_summary = pd.DataFrame([{
    "chunk":   i+1,
    "source":  d.metadata.get("source", ""),
    "h2":      d.metadata.get("section_h2", "-"),
    "h3":      d.metadata.get("section_h3", "-"),
    "chars":   len(d.page_content),
} for i, d in enumerate(all_docs)])

chunk_summary

In [8]:
# Summary table of all chunks
rows = []
for i in range(len(all_docs)):
    doc = all_docs[i]
    row = {
        "chunk":  i + 1,
        "source": doc.metadata.get("source", ""),
        "h2":     doc.metadata.get("section_h2", "-"),
        "h3":     doc.metadata.get("section_h3", "-"),
        "chars":  len(doc.page_content),
    }
    rows.append(row)

chunk_summary = pd.DataFrame(rows)
chunk_summary

,chunk,source,h2,h3,chars
0,1,01_payment_communication_policy.md,Overview,-,347
1,2,01_payment_communication_policy.md,Trigger Timeline,-,685
2,3,01_payment_communication_policy.md,General Principles,-,679
3,4,01_payment_communication_policy.md,Customer Risk Tiers,Low Risk (late payment ratio < 20%),272
4,5,01_payment_communication_policy.md,Customer Risk Tiers,Medium Risk (late payment ratio 20%–60%),167
5,6,01_payment_communication_policy.md,Customer Risk Tiers,High Risk (late payment ratio > 60%),212
6,7,01_payment_communication_policy.md,Payment Channels,-,277
7,8,02_email_templates.md,-,-,47
8,9,02_email_templates.md,Stage 1: Friendly Reminder (7 days before due ...,-,743
9,10,02_email_templates.md,Stage 2: Final Reminder (3 days before due date),-,614


In [9]:
import shutil

# Delete existing ChromaDB to remove duplicates
shutil.rmtree(CHROMA_PATH, ignore_errors=True)
CHROMA_PATH.mkdir(parents=True, exist_ok=True)

print("ChromaDB cleared — ready for clean ingest.")

ChromaDB cleared — ready for clean ingest.


## Store embeddings in ChromaDB

We have:
- An embedder (Gemini)
- Our playbook split into chunks

Now we store the chunks and their embeddings in ChromaDB — a vector store that persists to disk.



In [10]:
from langchain_chroma import Chroma

CHROMA_PATH.mkdir(parents=True, exist_ok=True)

vector_store = Chroma(
    collection_name="playbook",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH),
)


# Only ingest if collection is empty — avoids duplicates on re-run
if vector_store._collection.count() == 0:
    document_ids = vector_store.add_documents(documents=all_docs)
    print("Added", len(document_ids), "chunks to ChromaDB.")
else:
    print(vector_store._collection.count())

print("Persisted to:", CHROMA_PATH)

# Add all chunks and get their IDs back
#document_ids = vector_store.add_documents(documents=all_docs)
#print(f"Added {len(document_ids)} chunks to ChromaDB.")
#print(f"Persisted to: {CHROMA_PATH}")

Added 34 chunks to ChromaDB.
Persisted to: ../data/chroma_db


In [11]:
# Inspect what was stored
stored = vector_store.get_by_ids(document_ids[:4])
for doc in stored:
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Section: {doc.metadata.get('section_h2', '-')}")
    print(f"Content: {doc.page_content[:100]}...")
    print()

Source: 01_payment_communication_policy.md
Section: Overview
Content: # Accounts Receivable — Payment Communication Playbook  
## Overview  
This document defines the sta...

Source: 01_payment_communication_policy.md
Section: Trigger Timeline
Content: ## Trigger Timeline  
| Days to / past due | Status        | Action                          |
|----...

Source: 01_payment_communication_policy.md
Section: General Principles
Content: ## General Principles  
- Always address the customer by name and reference the specific invoice num...

Source: 01_payment_communication_policy.md
Section: Customer Risk Tiers
Content: ## Customer Risk Tiers  
### Low Risk (late payment ratio < 20%)
- Assume good faith. Payment delays...



## Use the vector store to retrieve similar chunks

Now we can query the vector store with a question and retrieve the most relevant playbook sections.

This is the **Retrieval** part of RAG.

In [12]:
query = (
    "Invoice overdue by 5 days. "
    "Customer late payment ratio is 10%,"  #which is medium risk between 20% and 60%. "
    "What communication action and tone should we use?"
)

retrieved_docs = vector_store.similarity_search(query, k=4)

print(f"Retrieved {len(retrieved_docs)} chunks:\n")
for doc in retrieved_docs:
    print(f"  [{doc.metadata.get('source')}] {doc.metadata.get('section_h2', '-')}")

Retrieved 4 chunks:

  [01_payment_communication_policy.md] Customer Risk Tiers
  [01_payment_communication_policy.md] Customer Risk Tiers
  [01_payment_communication_policy.md] Customer Risk Tiers
  [01_payment_communication_policy.md] General Principles


In [13]:
print(f"Total docs in ChromaDB: {vector_store._collection.count()}")

Total docs in ChromaDB: 34


In [14]:
# Read the content of the retrieved chunks
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1}: {doc.metadata.get('section_h2', '-')} ---")
    display(Markdown(doc.page_content))


--- Chunk 1: Customer Risk Tiers ---


### Medium Risk (late payment ratio 20%–60%)
- Neutral professional tone.
- Clearly state the due amount and due date.
- Request confirmation of expected payment date.


--- Chunk 2: Customer Risk Tiers ---


### High Risk (late payment ratio > 60%)
- Firm but professional tone.
- Reference payment history indirectly: "as per our agreed payment terms".
- Request immediate payment or a written payment commitment.  
---


--- Chunk 3: Customer Risk Tiers ---


## Customer Risk Tiers  
### Low Risk (late payment ratio < 20%)
- Assume good faith. Payment delays are likely administrative.
- Use collaborative language: "we wanted to make sure this didn't slip through".
- Offer to resend invoice or confirm payment details if needed.


--- Chunk 4: General Principles ---


## General Principles  
- Always address the customer by name and reference the specific invoice number.
- Keep tone professional and respectful at all times, regardless of how overdue the invoice is.
- Never threaten legal action before the +45 day final notice.
- Always include the exact amount due, the original due date, and payment instructions in every communication.
- For high-value invoices (above $50,000), escalate one stage earlier than the standard timeline.
- Customers with a late payment ratio below 10% should receive a softer tone at every stage.
- Customers with a late payment ratio above 60% should receive a firmer tone from the first overdue notice.  
---

This concludes the Retrieval part: we can now find the playbook sections most relevant to any invoice situation.

## Generate an action script with Gemini

Now we combine the retrieved chunks with the invoice data and ask Gemini to generate a structured action.



from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GEMINI_API_KEY,
    temperature=0.2,        # low = more consistent, structured output
    max_output_tokens=1500,
)

In [15]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0.2,
    max_tokens=1500,
)

print("LLM ready — Groq llama-3.1-8b-instant")

LLM ready — Groq llama-3.1-8b-instant


Now build our custom prompt template.

It tells Gemini to act as an AR assistant, provides the playbook context, the invoice data, and instructs it to respond in JSON:

In [16]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["context", "invoice_context"],
    template="""
You are a cash flow management assistant for a B2B supplier company.
Your job is to decide the correct communication action for an outstanding invoice,
based strictly on the company playbook provided below.

The goal is to maximise on-time cash collection while preserving customer relationships.

You must respond ONLY with a valid JSON object. No prose before or after the JSON.
Do not include markdown code fences.

The JSON must follow this exact schema:
{{
  "action": "send_email" | "no_action" | "manual_review",
  "stage": "stage_1_friendly" | "stage_2_final_reminder" | "stage_3_due_today" |
            "stage_4_first_overdue" | "stage_5_second_overdue" |
            "stage_6_escalation" | "stage_7_final_notice" | "none",
  "tone": "friendly" | "neutral" | "firm",
  "priority": "low" | "medium" | "high" | "critical",
  "subject": "<email subject line>",
  "email_body": "<full email body with placeholders filled in>",
  "reasoning": "<1-2 sentence explanation of why this action was chosen>",
  "playbook_reference": "<which playbook section justifies this decision>"
}}

 TONE RULES — YOU MUST FOLLOW THESE WHEN WRITING THE EMAIL BODY 

The tone field and the email_body MUST match. Do not write the same generic email regardless of tone.

FRIENDLY tone (low-risk customers, late_ratio < 20%):
- Warm, collaborative, assume good faith.
- Use phrases like: "we wanted to make sure this didn't slip through", "please let us know if you need anything".
- Offer to resend the invoice or confirm payment details.
- No urgency language.

NEUTRAL tone (medium-risk customers, late_ratio 20%-60%):
- Professional and factual.
- Clearly state the amount due and due date.
- Request confirmation of expected payment date.
- No emotional language in either direction.

FIRM tone (high-risk customers, late_ratio > 60%, or Stage 5+):
- Direct and firm, but always professional. Never rude.
- Reference agreed payment terms: "as per our agreed payment terms".
- Request immediate payment or a written payment commitment.
- Make consequences clear without threatening legal action before Stage 7.


IMPORTANT: Write the email_body as plain text only. No markdown, no bold, no dashes, no bullet points. Use line breaks only.



 PLAYBOOK CONTEXT 
{context}

 INVOICE CONTEXT
{invoice_context}

Generate the correct action JSON.
""".strip()
)

print("Prompt template ready.")

Prompt template ready.


# TEST QUERY


In [17]:
from langchain_core.output_parsers import StrOutputParser

# Build the context from retrieved chunks
docs_content = "\n\n---\n\n".join(
    f"[{doc.metadata.get('source')} — {doc.metadata.get('section_h2', '')}]\n{doc.page_content}"
    for doc in retrieved_docs
)

# Simple invoice context for testing
invoice_context = """
Invoice ID:           INV-002
Customer name:        SYSCO corporation
Amount due:           $47,200.00 USD
Due date:             2020-05-17
Days past due:        5 (overdue)
Business segment:     Food Distribution
Customer late ratio:  42%
Customer risk tier:   low
N previous invoices:  28
""".strip()

In [18]:
# See exactly what is being sent to the LLM
print("CONTEXT")
print(docs_content)

print("INVOICE CONTEXT")
print(invoice_context)

print("FULL PROMPT")
filled_prompt = prompt_template.format(
    context=docs_content,
    invoice_context=invoice_context,
)
print(filled_prompt)

CONTEXT
[01_payment_communication_policy.md — Customer Risk Tiers]
### Medium Risk (late payment ratio 20%–60%)
- Neutral professional tone.
- Clearly state the due amount and due date.
- Request confirmation of expected payment date.

---

[01_payment_communication_policy.md — Customer Risk Tiers]
### High Risk (late payment ratio > 60%)
- Firm but professional tone.
- Reference payment history indirectly: "as per our agreed payment terms".
- Request immediate payment or a written payment commitment.  
---

---

[01_payment_communication_policy.md — Customer Risk Tiers]
## Customer Risk Tiers  
### Low Risk (late payment ratio < 20%)
- Assume good faith. Payment delays are likely administrative.
- Use collaborative language: "we wanted to make sure this didn't slip through".
- Offer to resend invoice or confirm payment details if needed.

---

[01_payment_communication_policy.md — General Principles]
## General Principles  
- Always address the customer by name and reference the speci

In [19]:

filled_prompt = prompt_template.format(
    context=docs_content,
    invoice_context=invoice_context,
)
print(filled_prompt)

You are a cash flow management assistant for a B2B supplier company.
Your job is to decide the correct communication action for an outstanding invoice,
based strictly on the company playbook provided below.

The goal is to maximise on-time cash collection while preserving customer relationships.

You must respond ONLY with a valid JSON object. No prose before or after the JSON.
Do not include markdown code fences.

The JSON must follow this exact schema:
{
  "action": "send_email" | "no_action" | "manual_review",
  "stage": "stage_1_friendly" | "stage_2_final_reminder" | "stage_3_due_today" |
            "stage_4_first_overdue" | "stage_5_second_overdue" |
            "stage_6_escalation" | "stage_7_final_notice" | "none",
  "tone": "friendly" | "neutral" | "firm",
  "priority": "low" | "medium" | "high" | "critical",
  "subject": "<email subject line>",
  "email_body": "<full email body with placeholders filled in>",
  "reasoning": "<1-2 sentence explanation of why this action was cho

In [20]:
# Build LangChain chain
chain = prompt_template | llm | StrOutputParser()

# Run
raw_output = chain.invoke({
    "context": docs_content,
    "invoice_context": invoice_context,
})

print("Raw output from Gemini:")
print(raw_output)


clean = raw_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

result = json.loads(clean)
display(Markdown(result["email_body"]))

Raw output from Gemini:
{
  "action": "send_email",
  "stage": "stage_2_final_reminder",
  "tone": "friendly",
  "priority": "low",
  "subject": "Friendly Reminder: Overdue Invoice INV-002",
  "email_body": "Dear SYSCO corporation team,\n\nWe wanted to make sure this didn't slip through and that you have all the necessary information to settle your outstanding invoice INV-002. The amount due is $47,200.00 USD, and the original due date was 2020-05-17. If you need any assistance or would like us to resend the invoice, please let us know.\n\nThank you for your prompt attention to this matter.\n\nBest regards,\n[Your Name]",
  "reasoning": "Friendly tone is suitable for low-risk customers with a late ratio below 20%. This is a final reminder before escalating to the next stage.",
  "playbook_reference": "Customer Risk Tiers (Low Risk)"
}


Dear SYSCO corporation team,

We wanted to make sure this didn't slip through and that you have all the necessary information to settle your outstanding invoice INV-002. The amount due is $47,200.00 USD, and the original due date was 2020-05-17. If you need any assistance or would like us to resend the invoice, please let us know.

Thank you for your prompt attention to this matter.

Best regards,
[Your Name]

In [21]:
# Parse the JSON output
clean = re.sub(r"^```(?:json)?\s*", "", raw_output.strip(), flags=re.MULTILINE)
clean = re.sub(r"\s*```$", "", clean, flags=re.MULTILINE)
result = json.loads(clean.strip())

print("Parsed result:")
pprint(result)

Parsed result:
{'action': 'send_email',
 'email_body': 'Dear SYSCO corporation team,\n'
               '\n'
               "We wanted to make sure this didn't slip through and that you "
               'have all the necessary information to settle your outstanding '
               'invoice INV-002. The amount due is $47,200.00 USD, and the '
               'original due date was 2020-05-17. If you need any assistance '
               'or would like us to resend the invoice, please let us know.\n'
               '\n'
               'Thank you for your prompt attention to this matter.\n'
               '\n'
               'Best regards,\n'
               '[Your Name]',
 'playbook_reference': 'Customer Risk Tiers (Low Risk)',
 'priority': 'low',
 'reasoning': 'Friendly tone is suitable for low-risk customers with a late '
              'ratio below 20%. This is a final reminder before escalating to '
              'the next stage.',
 'stage': 'stage_2_final_reminder',
 'subject': 'Friendl

In [22]:
# Display the email body
print(f"Action:   {result['action']}")
print(f"Stage:    {result['stage']}")
print(f"Tone:     {result['tone']}")
print(f"Priority: {result['priority']}")
print(f"\nSubject: {result['subject']}")
print(f"\nReasoning: {result['reasoning']}")
print(f"\n--- EMAIL BODY ---")
display(Markdown(result["email_body"]))

Action:   send_email
Stage:    stage_2_final_reminder
Tone:     friendly
Priority: low

Subject: Friendly Reminder: Overdue Invoice INV-002

Reasoning: Friendly tone is suitable for low-risk customers with a late ratio below 20%. This is a final reminder before escalating to the next stage.

--- EMAIL BODY ---


Dear SYSCO corporation team,

We wanted to make sure this didn't slip through and that you have all the necessary information to settle your outstanding invoice INV-002. The amount due is $47,200.00 USD, and the original due date was 2020-05-17. If you need any assistance or would like us to resend the invoice, please let us know.

Thank you for your prompt attention to this matter.

Best regards,
[Your Name]

**We have our first RAG-generated action script — the LLM used the playbook to decide what to do.**

## Build reusable functions

Now let's refactor into clean reusable functions — `build_invoice_context()` and `generate_script()` — so we can run the pipeline on any invoice.

**This block is responsible for:**

Receiving an invoice

Building context about that invoice

Retrieving relevant chunks from the playbook stored in the vector store

Sending that context to the LLM

Asking the LLM for a structured JSON response

Validating whether that JSON matches the expected format

Falling back to a safe default if anything goes wrong

Returning a final decision that is ready to use

# Validation constants



The LLM should not return any random structure. It must follow a strict schema so the rest of the pipeline can trust the output.

In [23]:
# Validation constants
VALID_ACTIONS    = {"send_email", "no_action", "manual_review"}
VALID_STAGES     = {
    "stage_1_friendly", "stage_2_final_reminder", "stage_3_due_today",
    "stage_4_first_overdue", "stage_5_second_overdue",
    "stage_6_escalation", "stage_7_final_notice", "none"
}
VALID_TONES      = {"friendly", "neutral", "firm"}
VALID_PRIORITIES = {"low", "medium", "high", "critical"}
REQUIRED_KEYS    = {"action", "stage", "tone", "priority", "subject", "email_body", "reasoning", "playbook_reference"}

**This function converts a customer’s late payment ratio into a simple risk label:**

-low

-medium

-high

In [24]:
def get_risk_tier(late_ratio: float) -> str:
    if late_ratio < 0.20:
        return "low"
    elif 0.20 <= late_ratio < 0.60:
        return "medium"
    else:
        return "high"

# Build invoice context

This function takes the raw invoice dictionary and turns it into a clean text block for the prompt.
-Invoice ID:           INV-1001

-Customer name:        ABC Corp

-Customer number:      C123

-Amount due:           $4,500.00 USD

-Due date:             2026-03-01

-Days past due:        12 (overdue)

-Business segment:     Retail

-Payment terms:        NET30

-Predicted is_late:    1

-Predicted days late:  18

-Customer late ratio:  70%

-Customer risk tier:   high

-Customer risk score:  0.82

-N previous invoices:  14

In [25]:
def build_invoice_context(invoice: dict) -> str:
    """Format invoice data and model predictions as a string for the prompt."""
    late_ratio = invoice.get("cust_late_ratio", 0.5)
    return f"""
Invoice ID:           {invoice["invoice_id"]}
Customer name:        {invoice["customer_name"]}
Customer number:      {invoice["cust_number"]}
Amount due:           ${invoice["total_open_amount"]:,.2f} {invoice.get("currency", "USD")}
Due date:             {invoice["due_in_date"]}
Days past due:        {invoice["days_past_due"]} ({"overdue" if invoice["days_past_due"] > 0 else "not yet due"})
Business segment:     {invoice.get("business_segment", "Unknown")}
Payment terms:        {invoice.get("cust_payment_terms", "Unknown")}
Predicted is_late:    {invoice.get("predicted_is_late", "Unknown")}
Predicted days late:  {invoice.get("predicted_days_late", "Unknown")}
Customer late ratio:  {late_ratio:.0%}
Customer risk tier:   {get_risk_tier(late_ratio)}
Customer risk score:  {invoice.get("cust_risk_score", "Unknown")}
N previous invoices:  {invoice.get("cust_n_transactions", "Unknown")}
""".strip()



# Output validation

This function checks whether the LLM response follows the schema rules.

**What it checks?**

All required keys exist

Enum values are valid

Important text fields are not empty

In [26]:
def validate_output(output: dict) -> tuple:
    missing = REQUIRED_KEYS - set(output.keys())
    if missing:
        return False, f"Missing keys: {missing}"
    if output["action"] not in VALID_ACTIONS:
        return False, f"Invalid action: {output['action']}"
    if output["stage"] not in VALID_STAGES:
        return False, f"Invalid stage: {output['stage']}"
    if output["tone"] not in VALID_TONES:
        return False, f"Invalid tone: {output['tone']}"
    if output["priority"] not in VALID_PRIORITIES:
        return False, f"Invalid priority: {output['priority']}"

    # subject and email_body only required when sending an email
    if output["action"] == "send_email":
        for key in ["subject", "email_body"]:
            if not isinstance(output[key], str) or len(output[key].strip()) < 5:
                return False, f"Field '{key}' is empty or too short."

    for key in ["reasoning", "playbook_reference"]:
        if not isinstance(output[key], str) or len(output[key].strip()) < 5:
            return False, f"Field '{key}' is empty or too short."

    return True, "OK"

# Fallback output

This function creates a safe default response when something goes wrong.

**When it is used?**

The LLM call fails

JSON parsing fails

Validation fails

In [27]:
def fallback_output(invoice: dict, reason: str) -> dict:
    """Fallback when LLM fails or returns invalid JSON. Routes to manual review."""
    return {
        "action":             "manual_review",
        "stage":              "none",
        "tone":               "neutral",
        "priority":           "high" if invoice.get("days_past_due", 0) > 10 else "medium",
        "subject":            f"Manual review required — Invoice {invoice.get('invoice_id')}",
        "email_body":         "Please review this invoice manually.",
        "reasoning":          f"Fallback triggered: {reason}",
        "playbook_reference": "N/A — fallback",
        "invoice_id":         invoice.get("invoice_id"),
        "customer_name":      invoice.get("customer_name"),
        "generated_at":       datetime.now().isoformat(),
        "retrieved_sections": [],
    }

In [28]:
def generate_script(invoice, vector_store, k=4):
    # Build retrieval query from raw invoice facts
    if invoice["days_past_due"] > 0:
        status = "overdue"
    else:
        status = "upcoming"

    query = (
        "Invoice " + status + " by " + str(abs(invoice["days_past_due"])) + " days. "
        "Amount: $" + f"{invoice['total_open_amount']:,.0f}" + ". "
        "Customer risk: " + get_risk_tier(invoice.get("cust_late_ratio", 0.5)) + ". "
        "What communication action should be taken?"
    )

    docs = vector_store.similarity_search(query, k=k)

    context = ""
    for doc in docs:
        source  = doc.metadata.get("source")
        section = doc.metadata.get("section_h2", "")
        context = context + "[" + source + " — " + section + "]\n" + doc.page_content + "\n\n---\n\n"

    # Call the LLM
    chain = prompt_template | llm | StrOutputParser()
    try:
        raw_text = chain.invoke({
            "context":         context,
            "invoice_context": build_invoice_context(invoice),
        })
    except Exception as e:
        return fallback_output(invoice, "LLM error: " + str(e))

    # Parse JSON
    try:
        clean = raw_text.strip()
        if clean.startswith("```"):
            clean = re.sub(r"^```(?:json)?\s*", "", clean, flags=re.MULTILINE)
            clean = re.sub(r"\s*```$", "", clean, flags=re.MULTILINE)
            clean = clean.strip()
        output = json.loads(clean)
    except json.JSONDecodeError as e:
        return fallback_output(invoice, "JSON parse error: " + str(e))

    # Validate
    is_valid, error = validate_output(output)
    if not is_valid:
        return fallback_output(invoice, "Validation failed: " + error)

    # Add metadata
    output["invoice_id"]     = invoice.get("invoice_id")
    output["customer_name"]  = invoice.get("customer_name")
    output["generated_at"]   = datetime.now().isoformat()

    retrieved_sections = []
    for doc in docs:
        section_label = doc.metadata.get("source") + " — " + doc.metadata.get("section_h2", "")
        retrieved_sections.append(section_label)
    output["retrieved_sections"] = retrieved_sections

    return output

print("Functions defined.")

Functions defined.


## Test with 5 invoice scenarios

We test across the full range of situations our model will encounter:
- Low risk customer, invoice not yet due
- Medium risk, 5 days overdue
- High risk, 15 days overdue, large amount
- New customer, 3 days overdue
- Critical escalation, 30 days overdue

In [29]:
test_invoices = [
    # Scenario 1: Low risk, 7 days before due
    {
        "invoice_id": "INV-001", "customer_name": "KROGER foundation",
        "cust_number": "0200752302", "total_open_amount": 28500.00,
        "due_in_date": "2020-05-29", "days_past_due": -7,
        "business_segment": "Grocery Retail", "cust_payment_terms": "NAA8",
        "currency": "USD", "cust_late_ratio": 0.12, "cust_risk_score": 0.15,
        "cust_n_transactions": 45, "predicted_is_late": 0, "predicted_days_late": -2,
    },
    # Scenario 2: Medium risk, 5 days overdue
    {
        "invoice_id": "INV-002", "customer_name": "SYSCO corporation",
        "cust_number": "0200714710", "total_open_amount": 47200.00,
        "due_in_date": "2020-05-17", "days_past_due": 5,
        "business_segment": "Food Distribution", "cust_payment_terms": "NAH4",
        "currency": "USD", "cust_late_ratio": 0.42, "cust_risk_score": 0.38,
        "cust_n_transactions": 28, "predicted_is_late": 1, "predicted_days_late": 7,
    },
    # Scenario 3: High risk, 15 days overdue, large amount
    {
        "invoice_id": "INV-003", "customer_name": "BURR corporation",
        "cust_number": "0200555117", "total_open_amount": 82000.00,
        "due_in_date": "2020-05-07", "days_past_due": 15,
        "business_segment": "Grocery Retail", "cust_payment_terms": "NAD1",
        "currency": "USD", "cust_late_ratio": 0.75, "cust_risk_score": 0.81,
        "cust_n_transactions": 12, "predicted_is_late": 1, "predicted_days_late": 18,
    },
    # Scenario 4: New customer, 3 days overdue
    {
        "invoice_id": "INV-004", "customer_name": "MAINES associates",
        "cust_number": "0200800001", "total_open_amount": 9800.00,
        "due_in_date": "2020-05-19", "days_past_due": 3,
        "business_segment": "Food Distribution", "cust_payment_terms": "NAM4",
        "currency": "USD", "cust_late_ratio": 0.33, "cust_risk_score": 0.25,
        "cust_n_transactions": 2, "predicted_is_late": 1, "predicted_days_late": 4,
    },
    # Scenario 5: Critical, 30 days overdue
    {
        "invoice_id": "INV-005", "customer_name": "STATER systems",
        "cust_number": "0200743996", "total_open_amount": 54300.00,
        "due_in_date": "2020-04-22", "days_past_due": 30,
        "business_segment": "Grocery Retail", "cust_payment_terms": "NAH4",
        "currency": "USD", "cust_late_ratio": 0.68, "cust_risk_score": 0.72,
        "cust_n_transactions": 20, "predicted_is_late": 1, "predicted_days_late": 35,
    },
]

print(f"{len(test_invoices)} test scenarios ready.")

5 test scenarios ready.


# Run all scenarios

In [30]:

results = []

for i, invoice in enumerate(test_invoices):
    print(f"\n{'='*60}")
    print(f"Scenario {i+1}: {invoice['customer_name']} — {invoice['invoice_id']}")
    print(f"Days past due: {invoice['days_past_due']} | "
          f"Amount: ${invoice['total_open_amount']:,.0f} | "
          f"Late ratio: {invoice['cust_late_ratio']:.0%}")
    print(f"{'='*60}")

    result = generate_script(invoice, vector_store)
    results.append(result)

    print(f"Action:    {result['action']}")
    print(f"Stage:     {result['stage']}")
    print(f"Tone:      {result['tone']}")
    print(f"Priority:  {result['priority']}")
    print(f"Subject:   {result['subject']}")
    print(f"Reasoning: {result['reasoning']}")
    print(f"Playbook:  {result['playbook_reference']}")
    print(f"\nRetrieved sections:")
    for s in result["retrieved_sections"]:
        print(f"  - {s}")
    print(f"\n--- EMAIL ---")
    display(Markdown(result["email_body"]))


Scenario 1: KROGER foundation — INV-001
Days past due: -7 | Amount: $28,500 | Late ratio: 12%
Action:    send_email
Stage:     stage_1_friendly
Tone:      friendly
Priority:  low
Subject:   Upcoming payment reminder — Invoice INV-001
Reasoning: The customer has a low risk tier and a late ratio below 20%, so a friendly tone is appropriate for this stage.
Playbook:  [02_email_templates.md — Stage 1: Friendly Reminder (7 days before due date)]

Retrieved sections:
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 02_email_templates.md — Stage 1: Friendly Reminder (7 days before due date)
  - 01_payment_communication_policy.md — General Principles
  - 01_payment_communication_policy.md — Customer Risk Tiers

--- EMAIL ---


Dear KROGER foundation,

We hope this message finds you well.

This is a friendly reminder that invoice INV-001 for $28,500.00 USD is due on 2020-05-29 — in 7 days.

Please find the invoice details below:
- Invoice number: INV-001
- Amount due: $28,500.00 USD
- Due date: 2020-05-29
- Payment terms: NAA8

If you have already arranged payment, please disregard this message. If you have any questions or need a copy of the invoice, do not hesitate to contact us.

Thank you for your continued business.

Kind regards,
Accounts Receivable
[COMPANY_NAME]
[AR_EMAIL] | [AR_PHONE]


Scenario 2: SYSCO corporation — INV-002
Days past due: 5 | Amount: $47,200 | Late ratio: 42%
Action:    send_email
Stage:     stage_2_final_reminder
Tone:      neutral
Priority:  medium
Subject:   Reminder: Overdue Invoice INV-002 for $47,200.00
Reasoning: This is the second reminder for an overdue invoice, and the customer is classified as medium risk with a late payment ratio of 42%. A neutral tone is suitable for this stage.
Playbook:  Medium Risk (late payment ratio 20%–60%)

Retrieved sections:
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 01_payment_communication_policy.md — General Principles
  - 01_payment_communication_policy.md — Customer Risk Tiers

--- EMAIL ---


Dear SYSCO corporation, we are writing to remind you that invoice INV-002 for $47,200.00 is now overdue. The original due date was 2020-05-17. Please confirm your expected payment date for this invoice. You can find the payment instructions and details on our website. If you need any assistance or have questions, please do not hesitate to contact us. Thank you for your prompt attention to this matter.


Scenario 3: BURR corporation — INV-003
Days past due: 15 | Amount: $82,000 | Late ratio: 75%
Action:    send_email
Stage:     stage_4_first_overdue
Tone:      firm
Priority:  high
Subject:   Overdue Invoice INV-003: Payment Due
Reasoning: The customer is high-risk with a late payment ratio of 75%, so a firm tone is justified. The invoice is overdue, and immediate payment is requested.
Playbook:  [01_payment_communication_policy.md — Customer Risk Tiers] High Risk (late payment ratio > 60%)

Retrieved sections:
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 01_payment_communication_policy.md — General Principles
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 01_payment_communication_policy.md — Customer Risk Tiers

--- EMAIL ---


Dear Burr corporation, we wanted to bring to your attention that invoice INV-003 for $82,000.00 USD is now overdue. As per our agreed payment terms, we kindly request immediate payment or a written payment commitment. Please confirm your expected payment date and arrange for payment at your earliest convenience. If you have any questions or concerns, please do not hesitate to contact us. We appreciate your prompt attention to this matter. Best regards, [Your Name]


Scenario 4: MAINES associates — INV-004
Days past due: 3 | Amount: $9,800 | Late ratio: 33%
Action:    send_email
Stage:     stage_2_final_reminder
Tone:      neutral
Priority:  medium
Subject:   Reminder: Overdue Invoice INV-004
Reasoning: This is a final reminder to the customer before escalating to the next stage. The neutral tone is suitable for a medium-risk customer with a late payment ratio of 33%.
Playbook:  Customer Risk Tiers, Medium Risk

Retrieved sections:
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 01_payment_communication_policy.md — General Principles
  - 01_payment_communication_policy.md — Customer Risk Tiers

--- EMAIL ---


Dear MAINES associates team,

We are writing to remind you that invoice INV-004 in the amount of $9,800.00 USD is now overdue. The original due date was 2020-05-19.

To confirm, please let us know your expected payment date for this invoice. If you need any assistance with payment or require a copy of the invoice, please do not hesitate to contact us.

Thank you for your prompt attention to this matter.

Best regards,
[Your Name]


Scenario 5: STATER systems — INV-005
Days past due: 30 | Amount: $54,300 | Late ratio: 68%
Action:    send_email
Stage:     stage_4_first_overdue
Tone:      firm
Priority:  high
Subject:   Overdue Invoice INV-005: Payment Request
Reasoning: The customer is high-risk due to a late payment ratio of 68%, and the invoice is overdue. A firm tone is necessary to request immediate payment.
Playbook:  [01_payment_communication_policy.md — Customer Risk Tiers] High Risk (late payment ratio > 60%)

Retrieved sections:
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 01_payment_communication_policy.md — General Principles
  - 01_payment_communication_policy.md — Customer Risk Tiers
  - 01_payment_communication_policy.md — Customer Risk Tiers

--- EMAIL ---


Dear STATER systems team,

We wanted to bring to your attention that invoice INV-005 for $54,300.00 USD is now overdue. As per our agreed payment terms, we kindly request immediate payment or a written payment commitment. Please confirm your expected payment date and arrangements.

If you need any assistance with payment or require a copy of the invoice, please don't hesitate to contact us.

Thank you for your prompt attention to this matter.

Best regards,
[Your Name]

## Analyse the results

In [31]:
# Summary table
summary = pd.DataFrame([{
    "invoice":       r["invoice_id"],
    "customer":      r["customer_name"],
    "days_past_due": test_invoices[i]["days_past_due"],
    "amount":        f"${test_invoices[i]['total_open_amount']:,.0f}",
    "late_ratio":    f"{test_invoices[i]['cust_late_ratio']:.0%}",
    "action":        r["action"],
    "stage":         r["stage"],
    "tone":          r["tone"],
    "priority":      r["priority"],
} for i, r in enumerate(results)])

summary

,invoice,customer,days_past_due,amount,late_ratio,action,stage,tone,priority
0,INV-001,KROGER foundation,-7,"$28,500",12%,send_email,stage_1_friendly,friendly,low
1,INV-002,SYSCO corporation,5,"$47,200",42%,send_email,stage_2_final_reminder,neutral,medium
2,INV-003,BURR corporation,15,"$82,000",75%,send_email,stage_4_first_overdue,firm,high
3,INV-004,MAINES associates,3,"$9,800",33%,send_email,stage_2_final_reminder,neutral,medium
4,INV-005,STATER systems,30,"$54,300",68%,send_email,stage_4_first_overdue,firm,high


In [32]:
# Check which sections of the playbook were used most
from collections import Counter

all_sections = [s for r in results for s in r["retrieved_sections"]]
section_counts = Counter(all_sections)

print("Most retrieved playbook sections across all scenarios:")
for section, count in section_counts.most_common():
    print(f"  {count}x  {section}")

Most retrieved playbook sections across all scenarios:
  14x  01_payment_communication_policy.md — Customer Risk Tiers
  5x  01_payment_communication_policy.md — General Principles
  1x  02_email_templates.md — Stage 1: Friendly Reminder (7 days before due date)


In [33]:
# Fallback check — did any scenario fail?
fallbacks = [(i+1, r) for i, r in enumerate(results)
             if "Fallback triggered" in r.get("reasoning", "")]

if fallbacks:
    print(f"WARNING: {len(fallbacks)} scenario(s) used fallback:")
    for i, r in fallbacks:
        print(f"  Scenario {i}: {r['reasoning']}")
else:
    print("All scenarios generated successfully — no fallbacks triggered.")

All scenarios generated successfully — no fallbacks triggered.


In [34]:
# Re-run only scenario 1 and print raw output
invoice = test_invoices[0]
query = f"Invoice not yet due in 7 days. Amount $28,500. Customer risk: low."
docs = vector_store.similarity_search(query, k=4)
context = "\n\n---\n\n".join(
    f"[{doc.metadata.get('source')} — {doc.metadata.get('section_h2', '')}]\n{doc.page_content}"
    for doc in docs
)
chain = prompt_template | llm | StrOutputParser()
raw = chain.invoke({"context": context, "invoice_context": build_invoice_context(invoice)})
print(raw)

{
  "action": "no_action",
  "stage": "none",
  "tone": "friendly",
  "priority": "low",
  "subject": "Friendly Reminder: Upcoming Invoice Due Date",
  "email_body": "Dear KROGER foundation team,\n\nwe wanted to make sure this didn't slip through - our invoice INV-001 is due on 2020-05-29. If you need anything or would like us to resend the invoice, please let us know.\n\nBest regards,\n[Your Name]",
  "reasoning": "The customer is low-risk and the invoice is not yet due, so a friendly reminder is sufficient.",
  "playbook_reference": "01_payment_communication_policy.md — Customer Risk Tiers (Low Risk)"
}


## [Optional] Filter retrieval by metadata

We can filter our playbook chunks by source file.

For example, if we only want to retrieve from the email templates:

In [35]:
query = "Invoice overdue by 5 days. What email should we send?"

# Filter to only retrieve from email templates file
filtered_docs = vector_store.similarity_search(
    query,
    k=4,
    filter={"source": "02_email_templates.md"}
)

print(f"Retrieved {len(filtered_docs)} chunks from email templates only:")
for doc in filtered_docs:
    print(f"  {doc.metadata.get('section_h2', '-')}")

Retrieved 4 chunks from email templates only:
  Stage 4: First Overdue Notice (5 days past due)
  Stage 5: Second Overdue Notice (15 days past due)
  Stage 1: Friendly Reminder (7 days before due date)
  Stage 3: Due Today Notification (0 days)


>**You can use this to build more targeted queries — for example always retrieve the decision rules + the relevant email template, rather than letting the retriever decide.**